# Data preprocessing

In [1]:
# Import packages and libraries
import pandas as pd

In [2]:
# Load dataset
cdc_places_county_df = pd.read_csv('../data/raw/cdc_places_county.csv')
census_acs5_2023_df = pd.read_csv('../data/raw/census_acs5_2023.csv')
epa_aqs_annual_2023_df = pd.read_csv('../data/raw/epa_aqs_annual_2023.csv')
fred_series_df = pd.read_csv('../data/raw/fred_series.csv')

# 1. Preprocess CDC PLACES County Data

In [3]:
# Ensure locationid (FIPS) is a 5-digit string
cdc_places_county_df['locationid'] = cdc_places_county_df['locationid'].astype(str).str.zfill(5)

# 2. Preprocess Census ACS5 2023 Data

In [4]:
# Mapping based on EDA findings
census_rename_map = {
    'B01003_001E': 'total_population',
    'B19013_001E': 'median_income',
    'B25077_001E': 'median_home_value',
    'B25064_001E': 'median_gross_rent',
    'B25071_001E': 'rent_pct_income',
    'B17001_002E': 'poverty_count',
    'B23025_003E': 'employed_pop',
    'B23025_005E': 'unemployed_pop',
    'B15003_001E': 'edu_total_pop',
    'B15003_022E': 'bachelors_degree',
    'B15003_023E': 'masters_degree',
    'B15003_024E': 'professional_degree',
    'B15003_025E': 'doctorate_degree',
    'B08013_001E': 'total_travel_time',
    'B08303_001E': 'total_commuters',
    'B25003_001E': 'total_housing_units',
    'B25003_002E': 'owner_occupied_units',
    'metropolitan statistical area/micropolitan statistical area': 'cbsa_code'
}
census_acs5_2023_df = census_acs5_2023_df.rename(columns=census_rename_map)
census_acs5_2023_df['cbsa_code'] = census_acs5_2023_df['cbsa_code'].astype(str)

# 3. Preprocess EPA AQS Annual 2023 Data

In [5]:
# EDA found nulls in pollutant standards and exceedance counts
# Fill null exceedance counts with 0 as it usually means no exceedance was recorded
cols_to_fill_zero = ['primary_exceedance_count', 'secondary_exceedance_count', 'exceptional_data_count']
for col in cols_to_fill_zero:
    if col in epa_aqs_annual_2023_df.columns:
        epa_aqs_annual_2023_df[col] = epa_aqs_annual_2023_df[col].fillna(0)

# Standardize geographic IDs
epa_aqs_annual_2023_df['state_code'] = epa_aqs_annual_2023_df['state_code'].astype(str).str.zfill(2)
epa_aqs_annual_2023_df['county_code'] = epa_aqs_annual_2023_df['county_code'].astype(str).str.zfill(3)
epa_aqs_annual_2023_df['fips'] = epa_aqs_annual_2023_df['state_code'] + epa_aqs_annual_2023_df['county_code']
epa_aqs_annual_2023_df['cbsa_code'] = epa_aqs_annual_2023_df['cbsa_code'].astype(str).str.replace('\.0$', '', regex=True)

<>:12: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:12: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
/var/folders/mc/dd932nwj3699v8qmnb0tcs0h0000gn/T/ipykernel_30604/4142636684.py:12: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  epa_aqs_annual_2023_df['cbsa_code'] = epa_aqs_annual_2023_df['cbsa_code'].astype(str).str.replace('\.0$', '', regex=True)


# 4. Preprocess FRED Series Data

In [6]:
# EDA found ~9% missing values in 'value'
fred_series_df['date'] = pd.to_datetime(fred_series_df['date'])
fred_series_df['value'] = pd.to_numeric(fred_series_df['value'], errors='coerce')

# Handle missing values via interpolation per series
fred_series_df = fred_series_df.sort_values(['series_id', 'date'])
fred_series_df['value'] = fred_series_df.groupby('series_id')['value'].transform(lambda x: x.interpolate(method='linear'))

# Drop remaining NaNs (at the beginning of series)
fred_series_df = fred_series_df.dropna(subset=['value'])

# 5. Save Preprocessed Data

In [7]:
import os
os.makedirs('../data/processed', exist_ok=True)

cdc_places_county_df.to_csv('../data/processed/cdc_places_county.csv', index=False)
census_acs5_2023_df.to_csv('../data/processed/census_acs5_2023.csv', index=False)
epa_aqs_annual_2023_df.to_csv('../data/processed/epa_aqs_annual_2023.csv', index=False)
fred_series_df.to_csv('../data/processed/fred_series.csv', index=False)

print("Preprocessing complete and files saved to data/processed/")

Preprocessing complete and files saved to data/processed/
